In [1]:
!pip install groq python-dotenv numpy tqdm datasets

In [2]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

In [3]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [4]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! I'm here to help with any math problems you might have. What's on your mind? Do you need help with a specific equation, or would you like me to generate a random problem for you to solve?


#### GSM8K 데이터셋 확인해보기

In [9]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [5]:
### 수정해도 됩니다!
def extract_final_answer(response: str):
    regex = r"(?:Answer:|Model response:)\s*\$?([0-9,]+)\b|([0-9,]+)\s*(meters|cups|miles|minutes)"
    matches = re.finditer(regex, response, re.MULTILINE)
    results = [match.group(1) if match.group(1) else match.group(2).replace(",", "") for match in matches]

    if len(results) == 0:
        additional_regex = r"\$?([0-9,]+)"
        additional_matches = re.finditer(additional_regex, response, re.MULTILINE)
        results.extend([match.group(1).replace(",", "") for match in additional_matches])

    return results[-1] if results else None

In [6]:
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer_raw = extract_final_answer(response)
            predicted_answer = None

        
            if predicted_answer_raw and isinstance(predicted_answer_raw, str):
                try:
                    # 쉼표 제거 후 float 변환 시도
                    predicted_answer = float(predicted_answer_raw.replace(",", ""))
                except ValueError:
                    # 숫자로 바꿀 수 없는 경우(빈 문자열 등) None 처리
                    predicted_answer = None
            
            # 정답 비교 로직 (None인 경우 오답 처리됨)
            is_correct = False
            if predicted_answer is not None:
                diff = abs(predicted_answer - correct_answer)
                is_correct = diff < 1e-5
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [7]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [8]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [14]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

  0%|          | 0/10 [00:00<?, ?it/s]

 50%|█████     | 5/10 [00:02<00:02,  1.74it/s]

Progress: [5/10]
Current Acc.: [80.00%]


100%|██████████| 10/10 [00:04<00:00,  2.33it/s]

Progress: [10/10]
Current Acc.: [70.00%]


In [21]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!

# 실험할 shot 리스트 정의
shots = [0, 3, 5]

for shot in shots:
    print(f"\n>>> Direct Prompting {shot}-shot 테스트를 시작합니다...")
    
    # 1. 해당 shot 수에 맞는 Direct 프롬프트 생성
    # construct_direct_prompt 함수는 입력받은 num_examples 만큼 예시를 포함한 프롬프트를 만듭니다.
    direct_prompt = construct_direct_prompt(num_examples=shot)
    
    # 2. 벤치마크 테스트 수행 (항상 num_samples=50)
    # run_benchmark_test 함수가 모델 호출과 채점(accuracy 계산)을 수행합니다.
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=direct_prompt,
        num_samples=50,  # 과제 명세에 따라 50개로 설정
        VERBOSE=False
    )
    
    # 3. 지정된 파일 이름 형식으로 결과 저장
    filename = f"direct_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"--- {filename} 저장 완료 (정확도: {accuracy:.2%}) ---")


>>> Direct Prompting 0-shot 테스트를 시작합니다...


 10%|█         | 5/50 [00:05<01:07,  1.49s/it]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:17<01:30,  2.26s/it]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [85.00%]


 90%|█████████ | 45/50 [03:34<00:24,  4.83s/it]

Progress: [45/50]
Current Acc.: [82.22%]


100%|██████████| 50/50 [03:59<00:00,  4.79s/it]

Progress: [50/50]
Current Acc.: [84.00%]
--- direct_prompting_5.txt 저장 완료 (정확도: 84.00%) ---


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [22]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )
    prompt = "Instruction:\nSolve the following mathematical questions by reasoning step-by-step. "
    prompt += "Provide the final answer at the end in the format: 'Answer: [number]'.\n" #TODO: 프롬프트를 작성해주세요!

    for i in range(num_examples):
        #TODO: CoT example을 만들어주세요!
        idx = sampled_indices[i]
        cur_question = train_dataset['question'][idx]
        full_answer = train_dataset['answer'][idx]

        # '####'를 기준으로 풀이 과정과 최종 정답 분리
        reasoning, final_answer = full_answer.split("####")

        # <<계산식>> 태그 제거하여 자연스러운 문장으로 변환
        clean_reasoning = re.sub(r"<<.*?>>", "", reasoning).strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:\n{clean_reasoning}\nAnswer: {final_answer.strip()}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [23]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!

# CoT 실험할 shot 리스트
cot_shots = [0, 3, 5]

for shot in cot_shots:
    print(f"\n>>> CoT Prompting {shot}-shot 테스트 시작...")
    
    # 1. CoT 프롬프트 생성
    cot_prompt = construct_CoT_prompt(num_examples=shot)
    
    # 2. 벤치마크 실행 (50개 샘플 고정) LLM 과제.pdf]
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=cot_prompt,
        num_samples=50,
        VERBOSE=False
    )
    
    # 3. 결과 저장 
    filename = f"CoT_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"--- {filename} 저장 완료 (정확도: {accuracy:.2%}) ---")


>>> CoT Prompting 0-shot 테스트 시작...


 10%|█         | 5/50 [00:02<00:22,  1.97it/s]

Progress: [5/50]
Current Acc.: [100.00%]


 20%|██        | 10/50 [00:05<00:23,  1.73it/s]

Progress: [10/50]
Current Acc.: [90.00%]


...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [06:02<00:32,  6.59s/it]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [06:40<00:00,  8.01s/it]

Progress: [50/50]
Current Acc.: [76.00%]
--- CoT_prompting_5.txt 저장 완료 (정확도: 76.00%) ---


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [14]:
def construct_my_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train
    # 안정성을 위해 랜덤 샘플링 유지하되, 고품질 예시 유도
    sampled_indices = random.sample(range(len(train_dataset)), num_examples)

    # 1. 90%를 기록한 CoT 0-shot의 검증된 지시문 사용
    prompt = "Instruction: Solve the following mathematical questions by reasoning step-by-step. "
    prompt += "Provide the final answer at the end in the format: 'Answer: [number]'.\n"

    for i in range(num_examples):
        idx = sampled_indices[i]
        q = train_dataset['question'][idx]
        full_a = train_dataset['answer'][idx]
        reasoning, ans = full_a.split("####")
        
        # 2. 예시 구조를 CoT 0-shot 지시사항과 100% 일치시켜 모델의 혼란 방지
        prompt += f"\nQuestion: {q}\nAnswer: {re.sub(r'<<.*?>>', '', reasoning).strip()}\nAnswer: {ans.strip()}\n"

    # 3. 질문: 불필요한 수식어 없이 깔끔하게 질문 입력
    prompt += "\nQuestion:\n{question}\nAnswer:"
    
    
    return prompt

In [22]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!

my_shots = [0]

for shot in my_shots:
    print(f"\n>>> My New Prompting {shot}-shot 테스트 시작...")
    
    # 본인이 설계한 프롬프트 생성
    my_prompt = construct_my_prompt(num_examples=shot)
    
    # 벤치마크 실행 (항상 num_samples=50) 
    results, accuracy = run_benchmark_test(
        dataset=gsm8k_test,
        prompt=my_prompt,
        num_samples=50,
        VERBOSE=False
    )
    
    # 파일명 명세
    filename = f"My_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)
    
    print(f"--- {filename} 저장 완료 (정확도: {accuracy:.2%}) ---")


>>> My New Prompting 0-shot 테스트 시작...


 10%|█         | 5/50 [00:03<00:25,  1.76it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:06<00:25,  1.60it/s]

Progress: [10/50]
Current Acc.: [80.00%]


...

Progress: [40/50]
Current Acc.: [87.50%]


 90%|█████████ | 45/50 [01:38<00:16,  3.35s/it]

Progress: [45/50]
Current Acc.: [86.67%]


100%|██████████| 50/50 [01:51<00:00,  2.23s/it]

Progress: [50/50]
Current Acc.: [88.00%]
--- My_prompting_0.txt 저장 완료 (정확도: 88.00%) ---


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!